# Lucas KP-to-QUBO without slack variables: simulated annealing

This notebook keeps the model from `KP2QUBO_Lucas_NoSlack_SA.py` and adds an interactive diagram. The x-axis contains all 16 configurations of the four item variables `x=(x0,x1,x2,x3)`. For each configuration, the paired bars show:

- orange: total profit;
- blue: total loaded weight.

The three sliders control the simulated-annealing runs (`num_reads`), sweeps per run (`num_sweeps`), and Lagrangian/penalty value `L`. The y-axis uses integer ticks spaced by one unit and is scaled to the larger of the maximum profit and maximum possible loaded weight. The best feasible state found by SA is highlighted.

No password, token, server configuration, or `.env` value is read by this notebook. In VS Code, select the project interpreter/kernel that contains `numpy`, `matplotlib`, `ipywidgets`, `pyqubo`, and `dwave-neal` (for example the existing `Python 3.13.3 (QUBO)` environment).

In [1]:
import sys
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

try:
    from neal import SimulatedAnnealingSampler
    from pyqubo import Binary, Constraint, Placeholder
except ImportError as exc:
    raise ImportError(
        "This notebook needs pyqubo and dwave-neal. In the active VS Code kernel run: %pip install pyqubo dwave-neal"
    ) from exc

print("Python kernel:", sys.executable)
print("Dependencies imported successfully.")

Python kernel: c:\CODEX_PROJECTS\ALGORITMIeCOMPLESSITA\GITHUB-ALeCO_QUBO\LEZIONI\.venv\Scripts\python.exe
Dependencies imported successfully.


In [2]:
# Reference KP instance from the attached Python file.
profits = (10, 10, 12, 18)
weights = (2, 4, 6, 9)
capacity = 16

x_names = tuple(f"x{i}" for i in range(len(profits)))
y_names = tuple(f"y{c}" for c in range(capacity + 1))

def build_bqm(lagrangian):
    """Build the no-slack, one-hot Lucas BQM for a given L."""
    x = [Binary(name) for name in x_names]
    y = [Binary(name) for name in y_names]

    profit = sum(value * variable for value, variable in zip(profits, x))
    one_hot_penalty = (1 - sum(y)) ** 2
    declared_weight = sum(c * variable for c, variable in enumerate(y))
    loaded_weight = sum(weight * variable for weight, variable in zip(weights, x))
    weight_penalty = (declared_weight - loaded_weight) ** 2

    penalty = Constraint(
        one_hot_penalty + weight_penalty,
        label="penalty",
    )
    hamiltonian = -profit + Placeholder("L") * penalty
    return hamiltonian.compile().to_bqm(feed_dict={"L": float(lagrangian)})

# All item-bit configurations are the categories on the x-axis.
configurations = list(product((0, 1), repeat=len(profits)))
configuration_labels = ["".join(map(str, config)) for config in configurations]
profit_values = np.array([sum(p * bit for p, bit in zip(profits, config)) for config in configurations])
weight_values = np.array([sum(w * bit for w, bit in zip(weights, config)) for config in configurations])
axis_max = int(max(profit_values.max(), weight_values.max()))

print("Configurations:", len(configurations))
print("Y-axis maximum:", axis_max)

Configurations: 16
Y-axis maximum: 50


In [3]:
def decode_sample(sample, energy):
    """Decode one SA sample using the same logic as the source script."""
    selected_items = [i for i, name in enumerate(x_names) if sample[name] == 1]
    selected_targets = [c for c, name in enumerate(y_names) if sample[name] == 1]

    total_profit = sum(profits[i] for i in selected_items)
    total_weight = sum(weights[i] for i in selected_items)
    one_hot = len(selected_targets) == 1
    target_matches_weight = one_hot and selected_targets[0] == total_weight
    feasible = one_hot and target_matches_weight and total_weight <= capacity
    configuration = "".join(str(int(sample[name])) for name in x_names)

    return {
        "configuration": configuration,
        "items": selected_items,
        "target_weights": selected_targets,
        "profit": total_profit,
        "weight": total_weight,
        "feasible": feasible,
        "energy": float(energy),
    }

def run_sa(num_reads, num_sweeps, lagrangian, seed=123):
    bqm = build_bqm(lagrangian)
    sampler = SimulatedAnnealingSampler()
    sampleset = sampler.sample(
        bqm,
        num_reads=int(num_reads),
        num_sweeps=int(num_sweeps),
        seed=seed,
    )
    records = [
        decode_sample(sample, energy)
        for sample, energy in sampleset.data(fields=["sample", "energy"])
    ]
    feasible_records = [record for record in records if record["feasible"]]
    best_feasible = min(feasible_records, key=lambda record: record["energy"]) if feasible_records else None
    first_state = records[0] if records else None
    return sampleset, records, feasible_records, best_feasible, first_state

In [4]:
def plot_configuration_bars(best_feasible=None):
    indices = np.arange(len(configurations))
    width = 0.38

    fig, ax = plt.subplots(figsize=(15, 7))
    ax.bar(
        indices - width / 2,
        profit_values,
        width,
        color="#F28E2B",
        label="Profit",
    )
    ax.bar(
        indices + width / 2,
        weight_values,
        width,
        color="#3776C2",
        label="Loaded weight",
    )

    if best_feasible is not None:
        best_index = configuration_labels.index(best_feasible["configuration"])
        ax.axvspan(
            best_index - 0.5,
            best_index + 0.5,
            color="#59A14F",
            alpha=0.18,
            label=f"Best feasible: x={best_feasible['configuration']}",
        )

    ax.axhline(capacity, color="#3776C2", linestyle="--", linewidth=1.2, label="Capacity W=16")
    ax.set_title("All item configurations: profit versus loaded weight")
    ax.set_xlabel("Configuration of optimization variables x=(x0,x1,x2,x3)")
    ax.set_ylabel("Numerical value (integer ticks, 1 unit)")
    ax.set_xticks(indices, [f"x={label}" for label in configuration_labels], rotation=45, ha="right")
    ax.set_ylim(0, axis_max)
    ax.set_yticks(np.arange(0, axis_max + 1, 1))
    ax.grid(axis="y", alpha=0.22)
    ax.legend(ncol=3, loc="upper left")
    fig.tight_layout()
    plt.show()

def refresh(num_reads, num_sweeps, lagrangian):
    sampleset, records, feasible_records, best_feasible, first_state = run_sa(
        num_reads=num_reads,
        num_sweeps=num_sweeps,
        lagrangian=lagrangian,
    )
    plot_configuration_bars(best_feasible)

    print(f"SA runs: {num_reads} | sweeps per run: {num_sweeps} | L: {lagrangian}")
    print(f"Feasible states found: {len(feasible_records)} / {len(sampleset)}")
    if best_feasible is None:
        print("No feasible state found. Increase runs, sweeps, or L.")
    else:
        print("Best feasible state:", best_feasible)
    print("First state returned by SA:", first_state)

runs_slider = widgets.IntSlider(
    value=100, min=1, max=500, step=1,
    description="SA runs", continuous_update=False,
    style={"description_width": "initial"},
)
sweeps_slider = widgets.IntSlider(
    value=1000, min=10, max=5000, step=10,
    description="SA sweeps", continuous_update=False,
    style={"description_width": "initial"},
)
lagrangian_slider = widgets.IntSlider(
    value=20, min=1, max=100, step=1,
    description="Lagrangian L", continuous_update=False,
    style={"description_width": "initial"},
)

controls = widgets.VBox([runs_slider, sweeps_slider, lagrangian_slider])
output = widgets.interactive_output(
    refresh,
    {
        "num_reads": runs_slider,
        "num_sweeps": sweeps_slider,
        "lagrangian": lagrangian_slider,
    },
)
display(controls, output)

Output()